## Imports and Functions

In [2]:
from pathlib import Path 

import cv2
import IPython
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go

In [3]:
def drop_duplicates(df):
    to_drop = []
    for name in df['name'].unique():
        t = df[df['name'] == name]
        for idx, row in t.iterrows():
            for i, r in t.iterrows():
                if (r['y1'] > row['y1'] and r['y1'] < row['y1']) and \
                (r['y2'] > row['y1'] and r['y2'] < row['y2']):
                    to_drop.append(row['name'])
    return list(set(to_drop))

    # to_drop = []
    # for name in dlib_status['name'].unique():
    #     t = dlib_status[dlib_status['name'] == name]
    #     for idx, row in t.iterrows():
    #         for i, r in t.iterrows():
    #             if (r['x1'] > row['x1'] and r['x1'] < row['x2']) and \
    #             (r['x2'] > row['x1'] and r['x2'] < row['x2']) and \
    #             (r['y1'] > row['y1'] and r['y1'] < row['y1']) and \
    #             (r['y2'] > row['y1'] and r['y2'] < row['y2']):
    #                 to_drop.append(row['name'])
    # list(set(to_drop))


In [4]:
def compare_faces(source, 
                  faces):
    temp = faces.rename({'x1': 'x1_ground',
            'y1': 'y1_ground',
            'x2': 'x2_ground',
            'y2': 'y2_ground',
            'width': 'width_ground',
            'height': 'height_ground'}, axis=1)
    temp = temp[['x1_ground',
                    'y1_ground',
                    'x2_ground',
                    'y2_ground',
                    'width_ground',
                    'height_ground',
                    'face_num',
                    'name']]

    df = source.merge(temp,
                on=['name', 'face_num'],
                how='left')
    df['status'] = df.apply(lambda x: 'tp' if pd.notna(x['x1_ground']) else 'fp', axis=1)

    rdf = temp.merge(df.drop(['x1_ground', 'y1_ground', 'x2_ground', 'y2_ground', 'width_ground', 'height_ground'], axis=1),
                 on=['name', 'face_num'],
                 how='left')
    rdf['status'] = rdf.apply(lambda x: 'fn' if pd.isna(x['x1']) else np.nan, axis=1)
    fn = rdf[rdf['x1'].isna()]

    df = pd.concat([df, fn], axis=0)
    
    # df['center_x'] = df.apply(lambda x: x['x1'] + int(x['width']/2), axis=1)
    # df['center_y'] = df.apply(lambda x: x['y1'] + int(x['height']/2), axis=1)
    # df['center_ground_x'] = df.apply(lambda x: x['x1_ground'] + int(x['width_ground']/2), axis=1)
    # df['center_ground_y'] = df.apply(lambda x: x['y1_ground'] + int(x['height_ground']/2), axis=1)
    
    return df

In [5]:
def show_image(image):
    _, ret = cv2.imencode('.jpg', image)
    i = IPython.display.Image(data=ret)
    IPython.display.display(i)

In [6]:
def show_by_id(id):
    faces = pd.read_csv('../data/faces.csv', index_col=0)
    fp = Path('../test_images').joinpath(faces.iloc[id]['name'])
    img = cv2.imread(str(fp))
    print(fp.name)
    show_image(img)

In [7]:
def show_correct_by_id(id, df):
    temp = df[(df['id'] == id) & (df['x1'].notna())]
    return temp['detector']

In [8]:
def show_by_name(name):
    fp = Path('../test_images').joinpath(name)
    img = cv2.imread(str(fp))
    show_image(img)

In [9]:
def show_heatmap(df):
    g = df[['status', 'name']].groupby('status').count()
    d = {'tp': round(g.loc['tp'].item()/g['name'].sum(), 3),
         'fp': round(g.loc['fp'].item()/g['name'].sum(), 3),
         'fn': round(g.loc['fn'].item()/g['name'].sum(), 3),
         'tn': 0}
    a = [[d['tp'], d['fp']], [d['fn'], d['tn']]]
    text = [[f'True Positive: {d["tp"]} ({g.loc["tp"].item()}/{g["name"].sum()})',
             f'False Positive: {d["fp"]} ({g.loc["fp"].item()}/{g["name"].sum()})'],
            [f'False Negative: {d["fn"]} ({g.loc["fn"].item()}/{g["name"].sum()})',
             f'True Negative: {d["tn"]} (0/{g["name"].sum()})']]
    fig = px.imshow(a, 
                    x=['positive', 'negative'],
                    y=['positive', 'negative'],
                    labels={'y': 'Predicted', 'x': 'Actual'})
    fig.update_traces(text=text, texttemplate="%{text}")
    fig.show()   

## Loading the Data

In [10]:
faces = pd.read_csv('../data/faces.csv', index_col=0)
print(faces.shape)
faces.head()

(1895, 12)


,name,img_height,img_width,x1,y1,x2,y2,width,height,area,pct_of_frame,face_num
0,Billions.S01E02.1080p.BluRay.x265-RARBG_40128.png,1080,1920,323,5,919,731,596,726,432696,0.208669,0
1,Billions.S01E09.1080p.BluRay.x265-RARBG_52704.png,1080,1920,262,42,614,507,352,465,163680,0.078935,0
2,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png,1080,1920,484,389,671,617,187,228,42636,0.020561,0
3,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png,1080,1920,1003,42,1195,261,192,219,42048,0.020278,1
4,Billions.S01E12.1080p.BluRay.x265-RARBG_78336.png,1080,1920,1121,66,1662,824,541,758,410078,0.197761,0


In [11]:
dlib_df = pd.read_csv('../test_results/dlib_new.csv', index_col=0)
print(dlib_df.shape)
dlib_df.head()

(1203, 13)


,x1,y1,x2,y2,width,height,area,confidence,face_num,img_width,img_height,pct_of_frame,name
0,530,251,938,659,408,408,166464,1.156534,0,1920,1080,0.080278,Billions.S01E02.1080p.BluRay.x265-RARBG_40128.png
1,239,181,522,464,283,283,80089,1.043341,0,1920,1080,0.038623,Billions.S01E09.1080p.BluRay.x265-RARBG_52704.png
2,546,450,683,586,137,136,18632,0.667519,0,1920,1080,0.008985,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png
3,996,109,1110,223,114,114,12996,0.637119,1,1920,1080,0.006267,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png
4,1045,296,1534,785,489,489,239121,1.075798,0,1920,1080,0.115317,Billions.S01E12.1080p.BluRay.x265-RARBG_78336.png


In [12]:
retina_df = pd.read_csv('../test_results/retina_new.csv', index_col=0)
# retina_df = retina_df.groupby(['name', 'face_num']).max()
# retina_df = retina_df.reset_index()
retina_df.head()

,x1,y1,x2,y2,width,height,area,confidence,face_num,img_width,img_height,pct_of_frame,name
0,534,190,926,712,392,522,204624,0.999803,0,1920,1080,0.098681,Billions.S01E02.1080p.BluRay.x265-RARBG_40128.png
1,276,104,528,491,252,387,97524,0.999671,0,1920,1080,0.047031,Billions.S01E09.1080p.BluRay.x265-RARBG_52704.png
2,1002,81,1107,262,105,181,19005,0.999594,0,1920,1080,0.009165,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png
3,543,411,668,598,125,187,23375,0.999471,1,1920,1080,0.011273,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png
4,660,276,676,296,16,20,320,0.965943,2,1920,1080,0.000154,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png


In [13]:
torch_df = pd.read_csv('../test_results/torch.csv', index_col=0)
torch_df.head()

,x1,y1,x2,y2,width,height,area,confidence,face_num,img_width,img_height,pct_of_frame,name
0,544,207,913,698,369,491,181179,698.003601,0,1920,1080,0.087374,Billions.S01E02.1080p.BluRay.x265-RARBG_40128.png
1,263,162,500,455,237,293,69441,455.585022,0,1920,1080,0.033488,Billions.S01E09.1080p.BluRay.x265-RARBG_52704.png
2,1114,225,1560,821,446,596,265816,821.530823,0,1920,1080,0.128191,Billions.S01E12.1080p.BluRay.x265-RARBG_78336.png
3,493,227,805,645,312,418,130416,645.647217,0,1920,1080,0.062894,Billions.S02E07.1080p.BluRay.x265-RARBG_69600.png
4,501,93,916,696,415,603,250245,696.220093,0,1920,1080,0.120681,Billions.S03E05.1080p.WEBRip.x265-RARBG_77832.png


In [14]:
cv_df = pd.read_csv('../test_results/cv.csv', index_col=0)
cv_df.head()

,x1,y1,x2,y2,img_width,img_height,width,height,area,pct_of_frame,confidence,face_num,id,name
0,522,167,912,681,1920,1080,390,514,200460,0.096672,0.995575,0,0,Billions.S01E02.1080p.BluRay.x265-RARBG_40128.png
1,276,113,505,477,1920,1080,229,364,83356,0.040199,0.942709,0,1,Billions.S01E09.1080p.BluRay.x265-RARBG_52704.png
2,1119,182,1542,799,1920,1080,423,617,260991,0.125864,0.999954,0,4,Billions.S01E12.1080p.BluRay.x265-RARBG_78336.png
3,970,207,1347,848,1920,1080,377,641,241657,0.116540,0.999548,0,5,Billions.S02E01.1080p.BluRay.x265-RARBG_32328.png
4,766,129,1200,749,1920,1080,434,620,269080,0.129765,0.995495,0,6,Billions.S02E07.1080p.BluRay.x265-RARBG_39816.png


In [15]:
batchface_df = pd.read_csv('../test_results/batchface.csv', index_col=0)
batchface_df.head()

,x1,y1,x2,y2,width,height,area,confidence,face_num,img_width,img_height,pct_of_frame,name,duration
0,534,157,923,722,389,565,219785,0.999868,0,1920,1080,0.105992,Billions.S01E02.1080p.BluRay.x265-RARBG_40128.png,2.901
1,276,127,524,490,248,363,90024,0.999628,0,1920,1080,0.043414,Billions.S01E09.1080p.BluRay.x265-RARBG_52704.png,0.133
2,1002,76,1099,258,97,182,17654,0.997610,0,1920,1080,0.008514,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png,0.143
3,535,396,664,601,129,205,26445,0.997312,1,1920,1080,0.012753,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png,0.143
4,661,275,678,298,17,23,391,0.969282,2,1920,1080,0.000189,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png,0.143


In [29]:
custom_df = pd.read_csv('../test_results/retina_custom_120.csv', index_col=0)
custom_df.head()

,x1,y1,x2,y2,width,height,area,confidence,face_num,img_width,img_height,pct_of_frame,name
0,59,19,102,79,43,60,2580,0.999794,0,1920,1080,0.001244,Billions.S01E02.1080p.BluRay.x265-RARBG_40128.png
1,30,11,58,54,28,43,1204,0.999600,0,1920,1080,0.000581,Billions.S01E09.1080p.BluRay.x265-RARBG_52704.png
2,60,47,74,66,14,19,266,0.999224,0,1920,1080,0.000128,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png
3,111,9,123,27,12,18,216,0.994894,1,1920,1080,0.000104,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png
4,123,22,172,90,49,68,3332,0.999045,0,1920,1080,0.001607,Billions.S01E12.1080p.BluRay.x265-RARBG_78336.png


## Comparing the Models

### Dlib

In [117]:
dlib_status = compare_faces(dlib_df, faces)

In [116]:
show_heatmap(dlib_status)

### RetinaFace

In [17]:
retina_status = compare_faces(retina_df, faces)
show_heatmap(retina_status)

#### Analysis

In [122]:
fig = px.histogram(retina_status, x='pct_of_frame', color='status', histnorm='percent')
fig.show()

In [123]:
fig = px.scatter(retina_status, x='pct_of_frame', y='confidence')
fig.show()

In [124]:
g = retina_status[['status', 'confidence']].groupby('status').mean()
fig = px.bar(g, x=g.index, y='confidence')
fig.show()

In [125]:
g = retina_status[['status', 'pct_of_frame']].groupby('status').mean()
fig = px.bar(g, x=g.index, y='pct_of_frame')
fig.show()

#### False Positives

In [24]:
temp = retina_status[retina_status['status'] == 'fp']['name'].unique().tolist()
print(len(temp))
[print(x) for x in temp]

103
Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png
Cheers.S02E06.1080p.BluRay.x265-RARBG_24360.png
Cheers.S03E05.1080p.BluRay.x265-RARBG_33960.png
Cheers.S03E05.1080p.BluRay.x265-RARBG_34080.png
Cheers.S03E06.1080p.BluRay.x265-RARBG_31200.png
Cheers.S03E07.1080p.BluRay.x265-RARBG_16704.png
Cheers.S03E18.1080p.BluRay.x265-RARBG_31848.png
Cheers.S03E25.1080p.BluRay.x265-RARBG_13920.png
Cheers.S04E15.1080p.BluRay.x265-RARBG_01104.png
Cheers.S04E23.1080p.BluRay.x265-RARBG_14064.png
Cheers.S05E21.1080p.BluRay.x265-RARBG_06696.png
Cheers.S06E08.1080p.BluRay.x265-RARBG_00120.png
Cheers.S06E09.1080p.BluRay.x265-RARBG_26112.png
Cheers.S06E20.1080p.BluRay.x265-RARBG_30912.png
Cheers.S06E22.1080p.BluRay.x265-RARBG_30504.png
Cheers.S07E08.1080p.BluRay.x265-RARBG_06768.png
Cheers.S08E19.1080p.BluRay.x265-RARBG_28848.png
Cheers.S09E06.1080p.BluRay.x265-RARBG_23304.png
Cheers.S09E23.1080p.BluRay.x265-RARBG_06960.png
Cheers.S10E19.1080p.BluRay.x265-RARBG_16776.png
Cheers.S11E13.1080p.BluRay.x265-RA

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

### BatchFace

In [37]:
batch_status = compare_faces(batchface_df, faces)
show_heatmap(batch_status)

In [27]:
batch_status[batch_status['status']=='fp'][['name', 'face_num', 'confidence']]

,name,face_num,confidence
4,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png,2,0.960874
8,Billions.S02E07.1080p.BluRay.x265-RARBG_39816.png,1,0.526186
17,Billions.S04E05.1080p.WEBRip.x265-RARBG_37200.png,1,0.820354
28,Billions.S05E08.1080p.WEBRip.x265-RARBG_72000.png,1,0.569218
40,Carnivale.S01E10.1080p.WEBRip.10Bit.EAC3.H265-...,1,0.653868
...,...,...,...
2459,Will.&.Grace.S11E06.Performance.Anxiety.1080p....,14,0.589030
2462,Will.&.Grace.S11E08.Lies.&.Whispers.1080p.Web....,2,0.995879
2463,Will.&.Grace.S11E08.Lies.&.Whispers.1080p.Web....,3,0.591159
2467,Will.&.Grace.S11E08.Lies.&.Whispers.1080p.Web....,3,0.911256


In [25]:
temp = batch_status[batch_status['status']=='fp']['name'].unique().tolist()
print(len(temp))
[print(x) for x in temp]

349
Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png
Billions.S02E07.1080p.BluRay.x265-RARBG_39816.png
Billions.S04E05.1080p.WEBRip.x265-RARBG_37200.png
Billions.S05E08.1080p.WEBRip.x265-RARBG_72000.png
Carnivale.S01E10.1080p.WEBRip.10Bit.EAC3.H265-d3g_11904.png
Carnivale.S02E03.1080p.WEBRip.10Bit.EAC3.H265-d3g_35616.png
Carnivale.S02E04.1080p.WEBRip.10Bit.EAC3.H265-d3g_32664.png
Carnivale.S02E09.1080p.WEBRip.10Bit.EAC3.H265-d3g_60816.png
Cheers.S01E02.1080p.BluRay.x265-RARBG_11184.png
Cheers.S01E09.1080p.BluRay.x265-RARBG_29496.png
Cheers.S01E22.1080p.BluRay.x265-RARBG_25800.png
Cheers.S02E06.1080p.BluRay.x265-RARBG_24360.png
Cheers.S02E20.1080p.BluRay.x265-RARBG_15768.png
Cheers.S03E01.1080p.BluRay.x265-RARBG_32352.png
Cheers.S03E05.1080p.BluRay.x265-RARBG_33960.png
Cheers.S03E05.1080p.BluRay.x265-RARBG_34080.png
Cheers.S03E06.1080p.BluRay.x265-RARBG_31200.png
Cheers.S03E07.1080p.BluRay.x265-RARBG_16704.png
Cheers.S03E07.1080p.BluRay.x265-RARBG_25800.png
Cheers.S03E18.1080p.BluRay.x

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

### RetinaCustom

In [30]:
custom_status = compare_faces(custom_df, faces)
show_heatmap(custom_status)

### MTCNN

In [119]:
torch_status = compare_faces(torch_df, faces)
show_heatmap(torch_status)

### OpenCV

In [120]:
cv_status = compare_faces(cv_df, faces)
show_heatmap(cv_status)

## Validation

In [28]:
temp = faces.merge(dlib_df,
                     on=['name', 'face_num'],
                     how='left')
temp

,name,img_height_x,img_width_x,x1_x,y1_x,x2_x,y2_x,width_x,height_x,area_x,...,y1_y,x2_y,y2_y,width_y,height_y,area_y,confidence,img_width_y,img_height_y,pct_of_frame_y
0,Billions.S01E02.1080p.BluRay.x265-RARBG_40128.png,1080,1920,323,5,919,731,596,726,432696,...,251.0,938.0,659.0,408.0,408.0,166464.0,1.156534,1920.0,1080.0,0.080278
1,Billions.S01E09.1080p.BluRay.x265-RARBG_52704.png,1080,1920,262,42,614,507,352,465,163680,...,181.0,522.0,464.0,283.0,283.0,80089.0,1.043341,1920.0,1080.0,0.038623
2,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png,1080,1920,484,389,671,617,187,228,42636,...,450.0,683.0,586.0,137.0,136.0,18632.0,0.667519,1920.0,1080.0,0.008985
3,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png,1080,1920,1003,42,1195,261,192,219,42048,...,109.0,1110.0,223.0,114.0,114.0,12996.0,0.637119,1920.0,1080.0,0.006267
4,Billions.S01E12.1080p.BluRay.x265-RARBG_78336.png,1080,1920,1121,66,1662,824,541,758,410078,...,296.0,1534.0,785.0,489.0,489.0,239121.0,1.075798,1920.0,1080.0,0.115317
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1890,Will.&.Grace.S11E08.Lies.&.Whispers.1080p.Web....,1080,1920,1612,261,1704,361,92,100,9200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1891,Will.&.Grace.S11E14.The.Favourite.1080p.Web.h2...,1080,1920,457,103,579,237,122,134,16348,...,228.0,859.0,307.0,79.0,79.0,6241.0,1.052727,1920.0,1080.0,0.003010
1892,Will.&.Grace.S11E14.The.Favourite.1080p.Web.h2...,1080,1920,778,223,868,330,90,107,9630,...,127.0,579.0,222.0,94.0,95.0,8930.0,1.017996,1920.0,1080.0,0.004307
1893,Will.&.Grace.S11E14.The.Favourite.1080p.Web.h2...,1080,1920,1240,204,1313,331,73,127,9271,...,228.0,1307.0,307.0,79.0,79.0,6241.0,0.669482,1920.0,1080.0,0.003010


In [29]:
dlib_df.shape

(1203, 13)

In [30]:
faces

,name,img_height,img_width,x1,y1,x2,y2,width,height,area,pct_of_frame,face_num
0,Billions.S01E02.1080p.BluRay.x265-RARBG_40128.png,1080,1920,323,5,919,731,596,726,432696,0.208669,0
1,Billions.S01E09.1080p.BluRay.x265-RARBG_52704.png,1080,1920,262,42,614,507,352,465,163680,0.078935,0
2,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png,1080,1920,484,389,671,617,187,228,42636,0.020561,0
3,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png,1080,1920,1003,42,1195,261,192,219,42048,0.020278,1
4,Billions.S01E12.1080p.BluRay.x265-RARBG_78336.png,1080,1920,1121,66,1662,824,541,758,410078,0.197761,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1890,Will.&.Grace.S11E08.Lies.&.Whispers.1080p.Web....,1080,1920,1612,261,1704,361,92,100,9200,0.004437,2
1891,Will.&.Grace.S11E14.The.Favourite.1080p.Web.h2...,1080,1920,457,103,579,237,122,134,16348,0.007884,0
1892,Will.&.Grace.S11E14.The.Favourite.1080p.Web.h2...,1080,1920,778,223,868,330,90,107,9630,0.004644,1
1893,Will.&.Grace.S11E14.The.Favourite.1080p.Web.h2...,1080,1920,1240,204,1313,331,73,127,9271,0.004471,2


In [31]:
dlib_df[dlib_df.duplicated(subset=['name', 'face_num'], keep=False)]

,x1,y1,x2,y2,width,height,area,confidence,face_num,img_width,img_height,pct_of_frame,name


In [32]:
temp = dlib_df.merge(faces,
                     on=['name', 'face_num'],
                     how='left')
temp.head()

,x1_x,y1_x,x2_x,y2_x,width_x,height_x,area_x,confidence,face_num,img_width_x,...,img_height_y,img_width_y,x1_y,y1_y,x2_y,y2_y,width_y,height_y,area_y,pct_of_frame_y
0,530,251,938,659,408,408,166464,1.156534,0,1920,...,1080.0,1920.0,323.0,5.0,919.0,731.0,596.0,726.0,432696.0,0.208669
1,239,181,522,464,283,283,80089,1.043341,0,1920,...,1080.0,1920.0,262.0,42.0,614.0,507.0,352.0,465.0,163680.0,0.078935
2,546,450,683,586,137,136,18632,0.667519,0,1920,...,1080.0,1920.0,484.0,389.0,671.0,617.0,187.0,228.0,42636.0,0.020561
3,996,109,1110,223,114,114,12996,0.637119,1,1920,...,1080.0,1920.0,1003.0,42.0,1195.0,261.0,192.0,219.0,42048.0,0.020278
4,1045,296,1534,785,489,489,239121,1.075798,0,1920,...,1080.0,1920.0,1121.0,66.0,1662.0,824.0,541.0,758.0,410078.0,0.197761


In [33]:
fp = temp[temp['x1_y'].isna()]
fp.shape

(6, 23)

In [34]:
temp.shape

(1203, 23)

In [35]:
eph = faces.merge(dlib_df,
            on=['name', 'face_num'],
            how='left')
eph[eph['x1_y'].isna()].shape

(698, 23)

In [36]:
eph[eph['x1_y'].isna()]

,name,img_height_x,img_width_x,x1_x,y1_x,x2_x,y2_x,width_x,height_x,area_x,...,y1_y,x2_y,y2_y,width_y,height_y,area_y,confidence,img_width_y,img_height_y,pct_of_frame_y
10,Billions.S04E02.1080p.WEBRip.x265-RARBG_52416.png,1080,1920,535,160,801,465,266,305,81130,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,Billions.S04E10.1080p.WEBRip.x265-RARBG_15288.png,1080,1920,1037,245,1238,483,201,238,47838,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,Billions.S04E10.1080p.WEBRip.x265-RARBG_15288.png,1080,1920,1500,383,1801,709,301,326,98126,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29,Billions.S06E08.1080p.WEBRip.x265-RARBG_27264.png,1080,1920,1130,119,1311,343,181,224,40544,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31,Billions.S06E10.1080p.WEBRip.x265-RARBG_73680.png,1080,1920,87,286,170,406,83,120,9960,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1883,Will.&.Grace.S11E06.Performance.Anxiety.1080p....,1080,1920,1529,213,1596,288,67,75,5025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1884,Will.&.Grace.S11E06.Performance.Anxiety.1080p....,1080,1920,1470,206,1523,280,53,74,3922,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1885,Will.&.Grace.S11E06.Performance.Anxiety.1080p....,1080,1920,1700,188,1763,272,63,84,5292,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1887,Will.&.Grace.S11E08.Lies.&.Whispers.1080p.Web....,1080,1920,1179,235,1307,389,128,154,19712,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
dlib_df

,x1,y1,x2,y2,width,height,area,confidence,face_num,img_width,img_height,pct_of_frame,name
0,530,251,938,659,408,408,166464,1.156534,0,1920,1080,0.080278,Billions.S01E02.1080p.BluRay.x265-RARBG_40128.png
1,239,181,522,464,283,283,80089,1.043341,0,1920,1080,0.038623,Billions.S01E09.1080p.BluRay.x265-RARBG_52704.png
2,546,450,683,586,137,136,18632,0.667519,0,1920,1080,0.008985,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png
3,996,109,1110,223,114,114,12996,0.637119,1,1920,1080,0.006267,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png
4,1045,296,1534,785,489,489,239121,1.075798,0,1920,1080,0.115317,Billions.S01E12.1080p.BluRay.x265-RARBG_78336.png
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1198,1164,164,1243,243,79,79,6241,0.990481,1,1920,1080,0.003010,Will.&.Grace.S11E08.Lies.&.Whispers.1080p.Web....
1199,780,228,859,307,79,79,6241,1.052727,0,1920,1080,0.003010,Will.&.Grace.S11E14.The.Favourite.1080p.Web.h2...
1200,485,127,579,222,94,95,8930,1.017996,1,1920,1080,0.004307,Will.&.Grace.S11E14.The.Favourite.1080p.Web.h2...
1201,1228,228,1307,307,79,79,6241,0.669482,2,1920,1080,0.003010,Will.&.Grace.S11E14.The.Favourite.1080p.Web.h2...


In [38]:
retina_status[retina_status['status'] == 'fp']['name'].unique().tolist()

['Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png',
 'Cheers.S02E06.1080p.BluRay.x265-RARBG_24360.png',
 'Cheers.S03E05.1080p.BluRay.x265-RARBG_33960.png',
 'Cheers.S03E05.1080p.BluRay.x265-RARBG_34080.png',
 'Cheers.S03E06.1080p.BluRay.x265-RARBG_31200.png',
 'Cheers.S03E07.1080p.BluRay.x265-RARBG_16704.png',
 'Cheers.S03E18.1080p.BluRay.x265-RARBG_31848.png',
 'Cheers.S03E25.1080p.BluRay.x265-RARBG_13920.png',
 'Cheers.S04E15.1080p.BluRay.x265-RARBG_01104.png',
 'Cheers.S04E23.1080p.BluRay.x265-RARBG_14064.png',
 'Cheers.S05E21.1080p.BluRay.x265-RARBG_06696.png',
 'Cheers.S06E08.1080p.BluRay.x265-RARBG_00120.png',
 'Cheers.S06E09.1080p.BluRay.x265-RARBG_26112.png',
 'Cheers.S06E20.1080p.BluRay.x265-RARBG_30912.png',
 'Cheers.S06E22.1080p.BluRay.x265-RARBG_30504.png',
 'Cheers.S07E08.1080p.BluRay.x265-RARBG_06768.png',
 'Cheers.S08E19.1080p.BluRay.x265-RARBG_28848.png',
 'Cheers.S09E06.1080p.BluRay.x265-RARBG_23304.png',
 'Cheers.S09E23.1080p.BluRay.x265-RARBG_06960.png',
 'Cheers.S

In [41]:
fp = retina_status[retina_status['status'] == 'fp']

In [87]:
temp = retina_df[retina_df['pct_of_frame'] > 0.01]

In [88]:
retina_status_test = compare_faces(temp, faces)

In [89]:
show_heatmap(retina_status_test)

In [ ]:
def compare_faces(source, 
                  faces):
    temp = faces.rename({'x1': 'x1_ground',
            'y1': 'y1_ground',
            'x2': 'x2_ground',
            'y2': 'y2_ground',
            'width': 'width_ground',
            'height': 'height_ground'}, axis=1)
    temp = temp[['x1_ground',
                    'y1_ground',
                    'x2_ground',
                    'y2_ground',
                    'width_ground',
                    'height_ground',
                    'face_num',
                    'name']]

    df = source.merge(temp,
                on=['name', 'face_num'],
                how='left')
    df['status'] = df.apply(lambda x: 'tp' if pd.notna(x['x1_ground']) else 'fp', axis=1)

    rdf = temp.merge(df.drop(['x1_ground', 'y1_ground', 'x2_ground', 'y2_ground', 'width_ground', 'height_ground'], axis=1),
                 on=['name', 'face_num'],
                 how='left')
    rdf['status'] = rdf.apply(lambda x: 'fn' if pd.isna(x['x1']) else np.nan, axis=1)
    fn = rdf[rdf['x1'].isna()]

    df = pd.concat([df, fn], axis=0)
    
    # df['center_x'] = df.apply(lambda x: x['x1'] + int(x['width']/2), axis=1)
    # df['center_y'] = df.apply(lambda x: x['y1'] + int(x['height']/2), axis=1)
    # df['center_ground_x'] = df.apply(lambda x: x['x1_ground'] + int(x['width_ground']/2), axis=1)
    # df['center_ground_y'] = df.apply(lambda x: x['y1_ground'] + int(x['height_ground']/2), axis=1)
    
    return df

In [106]:
temp = dlib_df.merge(faces,
                       on=['name', 'face_num'],
                       how='left')
temp

,x1_x,y1_x,x2_x,y2_x,width_x,height_x,area_x,confidence,face_num,img_width_x,...,img_height_y,img_width_y,x1_y,y1_y,x2_y,y2_y,width_y,height_y,area_y,pct_of_frame_y
0,530,251,938,659,408,408,166464,1.156534,0,1920,...,1080.0,1920.0,323.0,5.0,919.0,731.0,596.0,726.0,432696.0,0.208669
1,239,181,522,464,283,283,80089,1.043341,0,1920,...,1080.0,1920.0,262.0,42.0,614.0,507.0,352.0,465.0,163680.0,0.078935
2,546,450,683,586,137,136,18632,0.667519,0,1920,...,1080.0,1920.0,484.0,389.0,671.0,617.0,187.0,228.0,42636.0,0.020561
3,996,109,1110,223,114,114,12996,0.637119,1,1920,...,1080.0,1920.0,1003.0,42.0,1195.0,261.0,192.0,219.0,42048.0,0.020278
4,1045,296,1534,785,489,489,239121,1.075798,0,1920,...,1080.0,1920.0,1121.0,66.0,1662.0,824.0,541.0,758.0,410078.0,0.197761
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1198,1164,164,1243,243,79,79,6241,0.990481,1,1920,...,1080.0,1920.0,1165.0,127.0,1268.0,255.0,103.0,128.0,13184.0,0.006358
1199,780,228,859,307,79,79,6241,1.052727,0,1920,...,1080.0,1920.0,457.0,103.0,579.0,237.0,122.0,134.0,16348.0,0.007884
1200,485,127,579,222,94,95,8930,1.017996,1,1920,...,1080.0,1920.0,778.0,223.0,868.0,330.0,90.0,107.0,9630.0,0.004644
1201,1228,228,1307,307,79,79,6241,0.669482,2,1920,...,1080.0,1920.0,1240.0,204.0,1313.0,331.0,73.0,127.0,9271.0,0.004471


In [108]:
fp = temp[temp['x1_y'].isna()]
print(f'({fp.shape[0]}/{temp.shape[0]}) = {fp.shape[0]/temp.shape[0]}')

(6/1203) = 0.004987531172069825


In [109]:
temp = faces.merge(dlib_df,
                   on=['name', 'face_num'],
                   how='left')

In [110]:
fn = temp[temp['x1_y'].isna()]
print(f'({fn.shape[0]}/{temp.shape[0]}) = {fn.shape[0]/temp.shape[0]}')

(698/1895) = 0.3683377308707124


In [113]:
fn['name'].unique().tolist()

['Billions.S04E02.1080p.WEBRip.x265-RARBG_52416.png',
 'Billions.S04E10.1080p.WEBRip.x265-RARBG_15288.png',
 'Billions.S06E08.1080p.WEBRip.x265-RARBG_27264.png',
 'Billions.S06E10.1080p.WEBRip.x265-RARBG_73680.png',
 'Billions.S06E12.1080p.WEBRip.x265-RARBG_10632.png',
 'Carnivale.S02E03.1080p.WEBRip.10Bit.EAC3.H265-d3g_35616.png',
 'Carnivale.S02E04.1080p.WEBRip.10Bit.EAC3.H265-d3g_20880.png',
 'Carnivale.S02E04.1080p.WEBRip.10Bit.EAC3.H265-d3g_32664.png',
 'Carnivale.S02E11.1080p.WEBRip.10Bit.EAC3.H265-d3g_49632.png',
 'Cheers.S01E02.1080p.BluRay.x265-RARBG_11184.png',
 'Cheers.S01E07.1080p.BluRay.x265-RARBG_09000.png',
 'Cheers.S01E09.1080p.BluRay.x265-RARBG_18408.png',
 'Cheers.S01E09.1080p.BluRay.x265-RARBG_29496.png',
 'Cheers.S01E20.1080p.BluRay.x265-RARBG_24480.png',
 'Cheers.S02E01.1080p.BluRay.x265-RARBG_07056.png',
 'Cheers.S02E01.1080p.BluRay.x265-RARBG_26040.png',
 'Cheers.S02E20.1080p.BluRay.x265-RARBG_15768.png',
 'Cheers.S03E03.1080p.BluRay.x265-RARBG_09624.png',
 'Chee

In [94]:
a = retina_df[retina_df['pct_of_frame'] > 0.005]
eph = a.merge(faces,
                                                         on=['name', 'face_num'],
                                                         how='left')

(1361, 23)

In [95]:
fp = eph[eph['x1_y'].isna()]
fp.shape

(34, 23)

In [96]:
34/1361

0.024981631153563555

In [97]:
152/1979

0.07680646791308741

In [100]:
eph['status'] = eph.apply(lambda x: 'tp' if not pd.isnull(x['x1_y']) else 'fp', axis=1)

In [101]:
g = eph.groupby('status').count()
g

,name,face_num,x1_x,y1_x,x2_x,y2_x,width_x,height_x,area_x,confidence,...,img_height_y,img_width_y,x1_y,y1_y,x2_y,y2_y,width_y,height_y,area_y,pct_of_frame_y
status,,,,,,,,,,,,,,,,,,,,,
fp,34,34,34,34,34,34,34,34,34,34,...,0,0,0,0,0,0,0,0,0,0
tp,1327,1327,1327,1327,1327,1327,1327,1327,1327,1327,...,1327,1327,1327,1327,1327,1327,1327,1327,1327,1327


In [102]:
temp = faces.merge(retina_df, 
                   on=['name', 'face_num'],
                   how='left')
temp

,name,img_height_x,img_width_x,x1_x,y1_x,x2_x,y2_x,width_x,height_x,area_x,...,y1_y,x2_y,y2_y,width_y,height_y,area_y,confidence,img_width_y,img_height_y,pct_of_frame_y
0,Billions.S01E02.1080p.BluRay.x265-RARBG_40128.png,1080,1920,323,5,919,731,596,726,432696,...,190.0,926.0,712.0,392.0,522.0,204624.0,0.999803,1920.0,1080.0,0.098681
1,Billions.S01E09.1080p.BluRay.x265-RARBG_52704.png,1080,1920,262,42,614,507,352,465,163680,...,104.0,528.0,491.0,252.0,387.0,97524.0,0.999671,1920.0,1080.0,0.047031
2,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png,1080,1920,484,389,671,617,187,228,42636,...,81.0,1107.0,262.0,105.0,181.0,19005.0,0.999594,1920.0,1080.0,0.009165
3,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png,1080,1920,1003,42,1195,261,192,219,42048,...,411.0,668.0,598.0,125.0,187.0,23375.0,0.999471,1920.0,1080.0,0.011273
4,Billions.S01E12.1080p.BluRay.x265-RARBG_78336.png,1080,1920,1121,66,1662,824,541,758,410078,...,186.0,1548.0,807.0,427.0,621.0,265167.0,0.999094,1920.0,1080.0,0.127878
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1890,Will.&.Grace.S11E08.Lies.&.Whispers.1080p.Web....,1080,1920,1612,261,1704,361,92,100,9200,...,274.0,1673.0,351.0,56.0,77.0,4312.0,0.997497,1920.0,1080.0,0.002079
1891,Will.&.Grace.S11E14.The.Favourite.1080p.Web.h2...,1080,1920,457,103,579,237,122,134,16348,...,231.0,860.0,324.0,73.0,93.0,6789.0,0.999080,1920.0,1080.0,0.003274
1892,Will.&.Grace.S11E14.The.Favourite.1080p.Web.h2...,1080,1920,778,223,868,330,90,107,9630,...,114.0,575.0,232.0,85.0,118.0,10030.0,0.998779,1920.0,1080.0,0.004837
1893,Will.&.Grace.S11E14.The.Favourite.1080p.Web.h2...,1080,1920,1240,204,1313,331,73,127,9271,...,215.0,1300.0,323.0,58.0,108.0,6264.0,0.994483,1920.0,1080.0,0.003021


In [104]:
fn = temp[temp['x1_y'].isna()]
fn

,name,img_height_x,img_width_x,x1_x,y1_x,x2_x,y2_x,width_x,height_x,area_x,...,y1_y,x2_y,y2_y,width_y,height_y,area_y,confidence,img_width_y,img_height_y,pct_of_frame_y
33,Billions.S06E10.1080p.WEBRip.x265-RARBG_73680.png,1080,1920,1531,217,1722,438,191,221,42211,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34,Billions.S06E10.1080p.WEBRip.x265-RARBG_73680.png,1080,1920,1484,280,1574,440,90,160,14400,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
39,Carnivale.S02E03.1080p.WEBRip.10Bit.EAC3.H265-...,1080,1920,742,42,1513,827,771,785,605235,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
66,Cheers.S01E20.1080p.BluRay.x265-RARBG_24480.png,1080,1448,0,451,135,679,135,228,30780,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
136,Cheers.S04E14.1080p.BluRay.x265-RARBG_07608.png,1080,1440,1294,430,1440,631,146,201,29346,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1600,White.Collar.S01E05.1080p.BluRay.x265-RARBG_09...,1080,1920,1716,381,1750,444,34,63,2142,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1606,White.Collar.S01E12.1080p.BluRay.x265-RARBG_53...,1080,1920,1356,251,1437,392,81,141,11421,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1637,White.Collar.S03E07.1080p.WEBRip.x265-RARBG_09...,1080,1920,1057,363,1163,483,106,120,12720,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1675,Will.&.Grace.S02E19.An.Affair.to.Forget.480p.W...,480,640,601,82,640,226,39,144,5616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [105]:
68/1895

0.035883905013192614

In [2]:
import pandas as pd 

df = pd.read_csv('../data/video_dimensions.csv', index_col=0)
df.head()

,fp,width,height,resolution,aspect_ratio,aspect_ratio_coded
0,/home/amos/media/tv/Manhunt.2024.S01E03.Let.th...,3840,1608,4k,2.388060,2.35
1,/home/amos/media/tv/Alone.S10E09.Splintered.10...,1920,1080,1080p,1.777778,1.85
2,/home/amos/media/tv/Alone.S10E02.1080p.WEB.h26...,1920,1080,1080p,1.777778,1.85
3,/home/amos/media/tv/Alone.S10E02.1080p.WEB.h26...,1920,1080,1080p,1.777778,1.85
4,/home/amos/media/tv/Constellation.S01E04.HDR.2...,3840,2160,4k,1.777778,1.85


In [5]:
g = df.groupby('resolution').max()
g['fp'].values

array(['/home/amos/media/tv/modern_family/Modern.Family.S11.1080p.DSNP.WEB-DL.DDP5.1.H.264-MIXED/Modern.Family.S11E18.Finale.Part.2.1080p.DSNP.WEB-DL.DDP5.1.H.264-ZigZag.mkv',
       '/home/amos/media/tv/Will.&.Grace.S01-S11.480p.1080p.WEB-DL.AAC.DDP5.1.h264-JBee/Season 07/Will.&.Grace.S07E09.Saving.Grace.Again.Part.2.480p.Web.h264-JBee.mkv',
       '/home/amos/media/tv/the_curse_2023/The.Curse.2023.S01E03.HDR.2160p.WEB.H265-ActivePlatinumCaracalOfAwe/the.curse.2023.s01e03.hdr.2160p.web.h265-activeplatinumcaracalofawe.mkv',
       '/home/amos/media/tv/The Simpsons.S01-S31.1080p.WEB-DL.BluRay.10bit.x265.HEVC-PHOCiS/Season 03/The.Simpsons.S03E01.Stark.Raving.Dad.720p.HDTV.10bit.x265.HEVC-PHOCiS.mkv'],
      dtype=object)